In [0]:
%sql
use catalog `formula1`; select * from `bronze`.`circuits` limit 100;

### Ingest circuits.csv file
1. Read the file using spark Dataframe Reader API
2. Add Metadata columns 
- source file
- ingestion timestamp
3. write to bronze delta table


In [0]:
%run ../00-common/01-environment-config

In [0]:
%run ../00-common/02-bronze_helper

In [0]:
source_file = f"{landing_forlder_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType())
])

In [0]:

circuits_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(circuits_schema)
    .load(source_file)
    .select("*", "_metadata")
)

In [0]:
display(circuits_df)

In [0]:
from pyspark.sql import functions as F

circuits_final_df = add_ingestion_metadata(circuits_df)

display(circuits_final_df)

### Write to bronze delta table

In [0]:
(
    circuits_final_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_name)
)

In [0]:
%sql 
SELECT * FROM formula1.bronze.circuits